In [ ]:
import numpy as np, pandas as pd, os, pickle, json, torch, torch.nn as nn
from pathlib import Path
from sklearn.metrics import mean_absolute_error

curr_path_file = Path("current_run.txt")
run_rel = curr_path_file.read_text().strip()
RUN_PATH = Path("experiments") / Path(run_rel)
RUN_PATH.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = RUN_PATH / "config.json"
CONFIG = json.load(open(CONFIG_PATH))
CACHE_DIR = RUN_PATH / "cache"

MODEL_NAME = "LSTM"
PRED_DIR = RUN_PATH / "predictions" / MODEL_NAME
RESULTS_DIR = RUN_PATH / "results" / MODEL_NAME
MODELS_DIR = RUN_PATH / "models" / MODEL_NAME
for p in [PRED_DIR, RESULTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

all_tags = CONFIG["tags"]
horizons = CONFIG["horizons"]
region_targets = CONFIG["regions"]
window = CONFIG["window"]

run_tags = ['P','PC','PCS','PCSB','PCSBI','PCSBIG']
tags = [t for t in run_tags if t in all_tags]

SEED = CONFIG.get("seed", 0)
torch.manual_seed(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.use_deterministic_algorithms(True)
torch.set_float32_matmul_precision('medium')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")
print(f"Running LSTM on {len(tags)} tags × {len(region_targets)} regions × {len(horizons)} horizons")
print(f"Experiment: {CONFIG.get('exp_name','')} / {CONFIG.get('run_stamp','')}")

class LSTMNet(nn.Module):
    def __init__(self, ch, hidden=32, num_layers=3, dropout=0.0):
        super().__init__()
        eff_dropout = dropout if num_layers > 1 else 0.0
        self.inp = nn.Linear(ch, hidden)
        self.rnn = nn.LSTM(hidden, hidden, num_layers=num_layers, batch_first=True, dropout=eff_dropout, bidirectional=True)
        self.head = nn.Sequential(nn.LayerNorm(hidden*2), nn.Linear(hidden*2, hidden*2), nn.ReLU(), nn.Linear(hidden*2, 1))
    def forward(self, x):
        h = self.inp(x)
        o, _ = self.rnn(h)
        y = self.head(o[:, -1])
        return y

def fit_seq_scaled(Xtr, ytr, Xva, ch, device):
    sx = (Xtr.reshape(-1, ch).mean(axis=0), Xtr.reshape(-1, ch).std(axis=0) + 1e-8)
    m, s = sx
    Xtr_s = ((Xtr.reshape(-1, ch) - m) / s).reshape(Xtr.shape)
    Xva_s = ((Xva.reshape(-1, ch) - m) / s).reshape(Xva.shape)

    ym, ys = ytr.mean(), ytr.std() + 1e-8
    ytr_s = (ytr - ym) / ys

    n_val = max(1, int(0.10 * len(Xtr_s)))
    X_train, X_val = Xtr_s[:-n_val], Xtr_s[-n_val:]
    y_train, y_val = ytr_s[:-n_val], ytr_s[-n_val:]

    tx = torch.tensor(X_train, dtype=torch.float32, device=device)
    ty = torch.tensor(y_train, dtype=torch.float32, device=device).unsqueeze(1)
    vx = torch.tensor(X_val, dtype=torch.float32, device=device)
    vy = torch.tensor(y_val, dtype=torch.float32, device=device).unsqueeze(1)

    mdl = LSTMNet(ch, hidden=32, num_layers=3, dropout=0.0).to(device)
    opt = torch.optim.AdamW(mdl.parameters(), 1e-4, weight_decay=1e-4)

    best_loss = 1e9
    patience_left = 5
    best_state = None
    for _ in range(120):
        mdl.train()
        pred = mdl(tx)
        loss = nn.functional.mse_loss(pred, ty)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
        opt.step()

        mdl.eval()
        with torch.no_grad():
            vpred = mdl(vx)
            vloss = nn.functional.mse_loss(vpred, vy)
        if vloss.item() < best_loss:
            best_loss = vloss.item()
            patience_left = 5
            best_state = {k: v.detach().cpu() for k, v in mdl.state_dict().items()}
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    mdl.load_state_dict(best_state)
    mdl.eval()
    with torch.no_grad():
        va_x = torch.tensor(Xva_s, dtype=torch.float32, device=device)
        pr_s = mdl(va_x).cpu().squeeze(1).numpy()
    pr = pr_s * ys + ym
    return pr

scores_mean_tag  = {t:{r:[np.inf]*len(horizons) for r in region_targets} for t in tags}
scores_std_tag   = {t:{r:[0.0]*len(horizons) for r in region_targets} for t in tags}

total = len(tags)*len(region_targets)*len(horizons)
done = 0
tick_every = 50

for tag in tags:
    seq_npz_path = CACHE_DIR / "features_seq" / f"{tag}.npz"
    folds_pkl_path = CACHE_DIR / "folds" / f"{tag}.pkl"

    if not seq_npz_path.exists() or not folds_pkl_path.exists():
        print(f"[Skip] Missing cache for tag {tag}")
        continue

    seq_npz = np.load(seq_npz_path, allow_pickle=False)
    folds_map = pickle.load(open(folds_pkl_path,"rb"))

    for reg in region_targets:
        all_rows = []
        for hi, h in enumerate(horizons):
            kb = f"{reg}__h{h}"
            x_key = f"{kb}__X"
            y_key = f"{kb}__y"
            idx_key = f"{kb}__idx"
            if x_key not in seq_npz or y_key not in seq_npz or idx_key not in seq_npz:
                done += 1
                if done % tick_every == 0 or done == total:
                    print(f"Progress {done}/{total}")
                continue

            Xs = seq_npz[x_key]
            ys = seq_npz[y_key]
            idx = pd.to_datetime(seq_npz[idx_key])

            if Xs.ndim != 3:
                done += 1
                if done % tick_every == 0 or done == total:
                    print(f"Progress {done}/{total}")
                continue

            ch = Xs.shape[2]
            folds = folds_map.get(kb, [])
            maes = []
            for fold_i, f in enumerate(folds):
                tr, va = f[0], f[1]
                pred = fit_seq_scaled(Xs[tr], ys[tr], Xs[va], ch, device)
                mae = mean_absolute_error(ys[va], pred)
                maes.append(mae)
                all_rows.append(pd.DataFrame({
                    "date": idx[va],
                    "model": MODEL_NAME,
                    "horizon": h,
                    "fold": fold_i,
                    "actual": ys[va],
                    "predicted": pred
                }))

            if maes:
                scores_mean_tag[tag][reg][hi] = float(np.mean(maes))
                scores_std_tag[tag][reg][hi] = float(np.std(maes))

            done += 1
            if done % tick_every == 0 or done == total:
                print(f"Progress {done}/{total}")

        if all_rows:
            out_path = PRED_DIR / f"{tag}__{reg}__cv.parquet"
            pd.concat(all_rows, ignore_index=True).to_parquet(out_path, index=False)

rows = []
for tag in tags:
    for reg in region_targets:
        for hi, h in enumerate(horizons):
            rows.append({
                "tag": tag,
                "region": reg,
                "model": MODEL_NAME,
                "horizon": h,
                "mae_mean": scores_mean_tag[tag][reg][hi],
                "mae_std": scores_std_tag[tag][reg][hi]
            })
pd.DataFrame(rows).to_csv(RESULTS_DIR / "scores_tag_all_cv.csv", index=False)

best = []
for reg in region_targets:
    for hi, h in enumerate(horizons):
        vals = [scores_mean_tag[t][reg][hi] for t in tags if np.isfinite(scores_mean_tag[t][reg][hi])]
        best.append({
            "region": reg,
            "model": MODEL_NAME,
            "horizon": h,
            "best_mae_mean": float(np.min(vals)) if len(vals) > 0 else np.inf
        })
pd.DataFrame(best).to_csv(RESULTS_DIR / "scores_best_cv.csv", index=False)

with open(RESULTS_DIR / "scores_pickle_cv.pkl","wb") as f:
    pickle.dump({"scores_mean_tag": scores_mean_tag, "scores_std_tag": scores_std_tag}, f)

print(f"{MODEL_NAME} results saved under: {RUN_PATH}")


Running on: cuda
Running LSTM on 6 tags × 19 regions × 9 horizons
Experiment: additive_tags_main_global_aug / 2026_01_26_123128
Progress 50/1026
Progress 100/1026
Progress 150/1026
Progress 200/1026
Progress 250/1026
Progress 300/1026
Progress 350/1026
Progress 400/1026
Progress 450/1026
Progress 500/1026
Progress 550/1026
Progress 600/1026
Progress 650/1026
Progress 700/1026
Progress 750/1026
Progress 800/1026
Progress 850/1026
Progress 900/1026
Progress 950/1026
Progress 1000/1026
Progress 1026/1026
LSTM results saved under: experiments\additive_tags_main_global_aug\2026_01_26_123128
